<a href="https://colab.research.google.com/github/mennahassan-lab/Library_Project/blob/main/EYOUTH_30911180200069_Library_Project_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# City Library After-School Program — End-of-Summer Reading Report
### Project Notebook

This notebook walks through all three tasks of the project, in order:

1. **Task 1 — Data Gathering and Combination**: five SQL questions, then combining
   the database, book catalog, and Reading Kickoff signups into one dataset.
2. **Task 2 — Data Integrity**: finding and resolving four data quality problems.
3. **Task 3 — Data Fairness**: comparing neighborhoods for fair representation.

Each stage's output files were also saved individually and are part of this
submission; this notebook is the single place that shows the full working
process end to end.

In [18]:
import sqlite3
import json
import pandas as pd
from bs4 import BeautifulSoup

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## Task 1 — Data Gathering and Combination

### Step 1: Explore the database

Before answering any question, it's worth simply looking at what each table
holds and how the tables relate: `members` and `checkouts` are linked by
`member_id`; `checkouts` and `books` are linked by `book_id`.

In [19]:
conn = sqlite3.connect("library db.db")
cur = conn.cursor()
for tname in ["members", "books", "checkouts"]:
    cur.execute(f"SELECT COUNT(*) FROM {tname}")
    print(tname, "->", cur.fetchone()[0], "rows")
    cur.execute(f"PRAGMA table_info({tname})")
    print("  columns:", [c[1] for c in cur.fetchall()])


members -> 80 rows
  columns: ['member_id', 'first_name', 'last_name', 'grade', 'neighborhood', 'membership_status', 'join_date']
books -> 32 rows
  columns: ['book_id', 'title', 'author']
checkouts -> 391 rows
  columns: ['checkout_id', 'member_id', 'book_id', 'checkout_date', 'return_date']


In [20]:
members = pd.read_sql("SELECT * FROM members", conn)
books_db = pd.read_sql("SELECT * FROM books", conn)
checkouts = pd.read_sql("SELECT * FROM checkouts", conn)
members.head(3)


,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05
1,1002,Fares,Saleh,9.0,Maadi,Active,None
2,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23


### Step 2: The five SQL questions

Each question below is answered directly against the database. The full
write-up (query + reasoning + result) for all five also lives in
`task1_sql_answers.txt`.

**Question 1 — How much is each member borrowing?** (includes members with zero checkouts, via a LEFT JOIN)

In [21]:
q1 = '''
SELECT m.member_id, m.first_name, m.last_name, COUNT(c.checkout_id) AS checkout_count
FROM members m
LEFT JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.first_name, m.last_name
ORDER BY m.member_id;
'''
df1 = pd.read_sql(q1, conn)
print("Rows:", len(df1), "| Members with zero checkouts:", (df1['checkout_count'] == 0).sum())
df1.head(10)


Rows: 80 | Members with zero checkouts: 18


,member_id,first_name,last_name,checkout_count
0,1001,Salma,Ibrahim,1
1,1002,Fares,Saleh,2
2,1003,Bassel,Hegazy,9
3,1004,Fares,Wahba,0
4,1005,Youssef,Halim,3
5,1006,Layla,Mansour,1
6,1007,Sherif,Fouad,3
7,1008,Ziad,Saleh,19
8,1009,Hassan,Saleh,1
9,1010,Nour,Nabil,18


**Question 2 — Which books match a chosen author pattern?** (chosen pattern: author names starting with 'A')

In [22]:
q2 = '''
SELECT book_id, title, author
FROM books
WHERE author LIKE 'A%'
ORDER BY author, title;
'''
df2 = pd.read_sql(q2, conn)
print("Rows:", len(df2))
df2


Rows: 6


,book_id,title,author
0,504,Rooftop Astronomers,Adel Roushdy
1,503,The Lantern Maker,Adel Roushdy
2,502,Desert Compass,Amina Darwish
3,501,The Silver Kite,Amina Darwish
4,505,Letters to the Nile,Aya Hafez
5,506,The Paper Boat Club,Aya Hafez


**Question 3 — What are the most popular books?** (top 5 by times borrowed)

In [23]:
q3 = '''
SELECT b.book_id, b.title, b.author, COUNT(c.checkout_id) AS times_borrowed
FROM checkouts c
JOIN books b ON c.book_id = b.book_id
GROUP BY b.book_id, b.title, b.author
ORDER BY times_borrowed DESC
LIMIT 5;
'''
df3 = pd.read_sql(q3, conn)
df3


,book_id,title,author,times_borrowed
0,501,The Silver Kite,Amina Darwish,57
1,507,Fossils and Fireflies,Dalia Serry,55
2,513,Circuits for Beginners,Galal Mounir,46
3,519,Kites Over Cairo,Jasmine Wahdan,38
4,525,Storms and Sailboats,Mahmoud Rafei,25


**Question 4 — Who are the most active readers?** (top 10 members by books borrowed)

In [24]:
q4 = '''
SELECT m.member_id, m.first_name, m.last_name, COUNT(c.checkout_id) AS books_borrowed
FROM members m
JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.first_name, m.last_name
ORDER BY books_borrowed DESC
LIMIT 10;
'''
df4 = pd.read_sql(q4, conn)
df4


,member_id,first_name,last_name,books_borrowed
0,1034,Aya,Wahba,25
1,1044,Sherif,Saleh,21
2,1008,Ziad,Saleh,19
3,1010,Nour,Nabil,18
4,1027,Mostafa,Fouad,18
5,1018,Ahmed,Shafik,17
6,1024,Youssef,Hegazy,17
7,1065,Adam,Fahmy,17
8,1030,Reem,Osman,16
9,1047,Sara,Rashad,16


**Question 5 — What does a neighborhood's activity look like further back in time?** (chosen neighborhood: Maadi; records 11–20, newest to oldest)

In [25]:
q5 = '''
SELECT c.checkout_id, c.member_id, c.book_id, c.checkout_date, c.return_date
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
WHERE m.neighborhood = 'Maadi'
ORDER BY c.checkout_date DESC
LIMIT 10 OFFSET 10;
'''
df5 = pd.read_sql(q5, conn)
df5


,checkout_id,member_id,book_id,checkout_date,return_date
0,9103,1003,513,2025-09-04,2025-09-25
1,9081,1017,502,2025-08-25,2025-09-17
2,9001,1008,501,2025-08-23,2025-08-28
3,9050,1003,521,2025-08-21,None
4,9085,1018,525,2025-08-19,2025-09-18
5,9055,1015,525,2025-08-08,2025-08-27
6,9023,1018,503,2025-08-04,2025-08-27
7,9004,1013,519,2025-07-27,2025-08-16
8,9051,1003,513,2025-07-22,2025-08-18
9,9010,1009,513,2025-07-22,2025-08-16


In [26]:
conn.close()
print("All five SQL answers, with full reasoning, are also saved in task1_sql_answers.txt,")
print("along with a written reflection on web-page vs. API data sources.")


All five SQL answers, with full reasoning, are also saved in task1_sql_answers.txt,
along with a written reflection on web-page vs. API data sources.


### Step 3: Combine the three sources into one dataset

**Stage 1 — Members + Checkouts (pandas only, no SQL).** Every checkout is kept,
matched to exactly the right member, and a running per-member checkout total
is added.

In [27]:
with open("books json.json") as f:
    books_catalog = pd.DataFrame(json.load(f))

with open("books html.html") as f:
    soup = BeautifulSoup(f, "html.parser")
table = soup.find("table")
rows_html = table.find_all("tr")
header = [th.get_text(strip=True) for th in rows_html[0].find_all("th")]
data = [[td.get_text(strip=True) for td in r.find_all("td")] for r in rows_html[1:]]
kickoff = pd.DataFrame(data, columns=header).rename(columns={
    "Member ID": "member_id", "Book ID": "book_id", "Checkout Date": "checkout_date",
})
kickoff["member_id"] = kickoff["member_id"].astype(int)
kickoff["book_id"] = kickoff["book_id"].astype(int)
print("books_catalog:", books_catalog.shape, "| kickoff signups:", kickoff.shape)


books_catalog: (32, 5) | kickoff signups: (26, 3)


In [28]:
stage1 = checkouts.merge(members, on="member_id", how="left", validate="many_to_one")
assert len(stage1) == len(checkouts), "no checkout should be gained or lost"
assert stage1["first_name"].isna().sum() == 0, "every DB checkout must match a real member"

stage1["member_total_checkouts"] = stage1.groupby("member_id")["checkout_id"].transform("count")
print("Stage 1 shape:", stage1.shape)
stage1.head(3)


Stage 1 shape: (391, 12)


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,member_total_checkouts
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,16
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,14
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,5


**Stage 2 — Book details.** Book information comes from two places: the DB's `books` table (title, author) and the JSON catalog (genre, pages, publication_year, publisher). Both are folded in here, keyed on `book_id`; the checkout count stays exactly the same — only information is being added.

In [29]:
book_details = books_db.merge(books_catalog, on="book_id", how="left", validate="one_to_one")

stage2 = stage1.merge(book_details, on="book_id", how="left", validate="many_to_one")
assert len(stage2) == len(stage1), "row count must not change in this stage"
assert stage2["title"].isna().sum() == 0, "every checkout must carry book details"
print("Stage 2 shape:", stage2.shape)
stage2.head(3)


Stage 2 shape: (391, 18)


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,member_total_checkouts,title,author,genre,pages,publication_year,publisher
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,16,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,14,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,5,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books


**Stage 3 — Reading Kickoff checkouts.** The Kickoff page has no `checkout_id` and no `return_date` (books aren't due back until end of summer), so both are built here, and the rows are reshaped to the same columns as the database checkouts before being appended.

In [30]:
kickoff = kickoff.reset_index(drop=True)
kickoff["checkout_id"] = ["RK" + str(i + 1).zfill(3) for i in range(len(kickoff))]
kickoff["return_date"] = pd.NA

kickoff_full = kickoff.merge(members, on="member_id", how="left", validate="many_to_one")
kickoff_full = kickoff_full.merge(book_details, on="book_id", how="left", validate="many_to_one")

print("Kickoff checkouts with no matching registered member:",
      kickoff_full["first_name"].isna().sum(),
      "-> member_ids:", sorted(kickoff_full.loc[kickoff_full['first_name'].isna(), 'member_id'].unique()))
print("(kept for now, resolved explicitly as a documented decision in Task 2)")

kickoff_full["source"] = "reading_kickoff"
stage2["source"] = "database"
kickoff_full = kickoff_full.reindex(columns=stage2.columns)

combined = pd.concat([stage2, kickoff_full], ignore_index=True)
combined["member_total_checkouts"] = combined.groupby("member_id")["checkout_id"].transform("count")

print("Final combined shape:", combined.shape, "(expected", len(checkouts), "+", len(kickoff), "=", len(checkouts) + len(kickoff), ")")
combined.to_csv("task1_combined_data.csv", index=False)
combined.head(3)


Kickoff checkouts with no matching registered member: 5 -> member_ids: [np.int64(1104), np.int64(1150), np.int64(1201)]
(kept for now, resolved explicitly as a documented decision in Task 2)
Final combined shape: (417, 19) (expected 391 + 26 = 417 )


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,member_total_checkouts,title,author,genre,pages,publication_year,publisher,source
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,16,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House,database
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,14,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,database
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,5,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books,database


In [31]:
print("Row count check:", len(combined))
print("Columns:", list(combined.columns))
print()
print("Missing values per column:")
print(combined.isna().sum())


Row count check: 417
Columns: ['checkout_id', 'member_id', 'book_id', 'checkout_date', 'return_date', 'first_name', 'last_name', 'grade', 'neighborhood', 'membership_status', 'join_date', 'member_total_checkouts', 'title', 'author', 'genre', 'pages', 'publication_year', 'publisher', 'source']

Missing values per column:
checkout_id                0
member_id                  0
book_id                    0
checkout_date              0
return_date               91
first_name                 5
last_name                  5
grade                     41
neighborhood               5
membership_status          5
join_date                 11
member_total_checkouts     0
title                      0
author                     0
genre                      0
pages                      0
publication_year          35
publisher                  0
source                     0
dtype: int64


## Task 2 — Data Integrity

Working from `task1_combined_data.csv`, four problems are found and resolved.
The full Data Integrity Report is saved separately as `integrity_report.docx`;
this section shows the underlying work.

### Problem 1: Duplicate records

In [32]:
df = pd.read_csv("task1_combined_data.csv")
print("Starting shape:", df.shape)

exact_dupes = df.duplicated(keep="first")
print("Exact duplicate rows (every column identical):", exact_dupes.sum())

# Confirm we're not about to remove records that only LOOK similar
similar_not_dupe = df[df.duplicated(subset=["member_id", "book_id"], keep=False)]
similar_not_dupe = similar_not_dupe[~similar_not_dupe.duplicated(keep=False)]
print("Records that share member+book but are genuinely different checkouts (kept):", len(similar_not_dupe))

df = df[~exact_dupes].reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)


Starting shape: (417, 19)
Exact duplicate rows (every column identical): 8
Records that share member+book but are genuinely different checkouts (kept): 180
Shape after removing duplicates: (409, 19)


### Problem 2: Records belonging to no registered member

In [33]:
unmatched_mask = df["first_name"].isna() & (df["source"] == "reading_kickoff")
unmatched_ids = sorted(df.loc[unmatched_mask, "member_id"].unique().tolist())
print("Checkouts referencing an unregistered member_id:", unmatched_mask.sum(), "-> ids:", unmatched_ids)

df = df[~unmatched_mask].reset_index(drop=True)
print("Shape after removing unmatched-member records:", df.shape)


Checkouts referencing an unregistered member_id: 5 -> ids: [1104, 1150, 1201]
Shape after removing unmatched-member records: (404, 19)


### Problem 3: Inconsistent values in text columns

In [34]:
print("neighborhood before:")
print(df["neighborhood"].value_counts(dropna=False))

neighborhood_map = {"maadi": "Maadi", "zamalek": "Zamalek", "nasr city": "Nasr City",
                     "heliopolis": "Heliopolis", "shubra": "Shubra"}
df["neighborhood"] = df["neighborhood"].str.strip().str.lower().map(neighborhood_map).fillna(df["neighborhood"])

print()
print("neighborhood after:")
print(df["neighborhood"].value_counts(dropna=False))


neighborhood before:
neighborhood
Nasr City     100
Maadi          95
Heliopolis     86
Zamalek        58
Shubra         34
Maadi          19
zamalek        10
NASR CITY       1
HELIOPOLIS      1
Name: count, dtype: int64

neighborhood after:
neighborhood
Maadi         114
Nasr City     101
Heliopolis     87
Zamalek        68
Shubra         34
Name: count, dtype: int64


In [35]:
print("membership_status before:")
print(df["membership_status"].value_counts(dropna=False))

status_map = {"active": "Active", "inactive": "Inactive"}
df["membership_status"] = df["membership_status"].str.strip().str.lower().map(status_map).fillna(df["membership_status"])

print()
print("membership_status after:")
print(df["membership_status"].value_counts(dropna=False))


membership_status before:
membership_status
Active      275
Inactive     53
active       45
inactive     31
Name: count, dtype: int64

membership_status after:
membership_status
Active      320
Inactive     84
Name: count, dtype: int64


### Problem 4: Missing values (decided column by column)

In [36]:
print("Missing values remaining before this step:")
print(df.isna().sum())
print()

# return_date: missing means "not returned yet" -- a real status, not an error
df["return_date"] = df["return_date"].fillna("Not Yet Returned")

# grade, join_date, publication_year: genuinely unrecorded at the source;
# left as missing rather than fabricated, since none feed the rest of the analysis
print("Final missing-value counts:")
print(df.isna().sum())


Missing values remaining before this step:
checkout_id                0
member_id                  0
book_id                    0
checkout_date              0
return_date               86
first_name                 0
last_name                  0
grade                     36
neighborhood               0
membership_status          0
join_date                  6
member_total_checkouts     0
title                      0
author                     0
genre                      0
pages                      0
publication_year          33
publisher                  0
source                     0
dtype: int64

Final missing-value counts:
checkout_id                0
member_id                  0
book_id                    0
checkout_date              0
return_date                0
first_name                 0
last_name                  0
grade                     36
neighborhood               0
membership_status          0
join_date                  6
member_total_checkouts     0
title           

In [38]:
print("Final cleaned shape:", df.shape)
print("Duplicate rows remaining:", df.duplicated().sum())
df.to_csv("task2_cleaned_data.csv", index=False)
df.head(3)


Final cleaned shape: (404, 19)
Duplicate rows remaining: 0


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,member_total_checkouts,title,author,genre,pages,publication_year,publisher,source
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,16,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House,database
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,14,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,database
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,5,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books,database


## Task 3 — Data Fairness

Comparing every neighborhood's share of members against its share of
checkouts, using the cleaned dataset. The full write-up is saved separately
as `fairness_reflection.docx`.

In [39]:
by_neighborhood = df.groupby("neighborhood").agg(
    member_count=("member_id", "nunique"),
    checkout_count=("checkout_id", "count"),
).reset_index()
by_neighborhood["member_share_pct"] = (100 * by_neighborhood["member_count"] / by_neighborhood["member_count"].sum()).round(1)
by_neighborhood["checkout_share_pct"] = (100 * by_neighborhood["checkout_count"] / by_neighborhood["checkout_count"].sum()).round(1)
by_neighborhood["gap_pct_points"] = (by_neighborhood["checkout_share_pct"] - by_neighborhood["member_share_pct"]).round(1)
by_neighborhood = by_neighborhood.sort_values("gap_pct_points")
by_neighborhood


,neighborhood,member_count,checkout_count,member_share_pct,checkout_share_pct,gap_pct_points
1,Maadi,20,114,30.8,28.2,-2.6
4,Zamalek,11,68,16.9,16.8,-0.1
2,Nasr City,16,101,24.6,25.0,0.4
3,Shubra,5,34,7.7,8.4,0.7
0,Heliopolis,13,87,20.0,21.5,1.5


In [41]:
print("Largest gap:", by_neighborhood.iloc[0]['neighborhood'],
      f"({by_neighborhood.iloc[0]['gap_pct_points']:+} points)")
print("All gaps are within +/-10 percentage points, so by the threshold used in the")
print("Data Fairness Reflection, no neighborhood in this dataset is under-represented.")


Largest gap: Maadi (-2.6 points)
All gaps are within +/-10 percentage points, so by the threshold used in the
Data Fairness Reflection, no neighborhood in this dataset is under-represented.


## Summary

- **Task 1**: 5 SQL questions answered (`task1_sql_answers.txt`); combined dataset
  built from all 3 sources (`task1_combined_data.csv`, 417 rows before cleaning).
- **Task 2**: 4 data quality problems resolved (`task2_cleaned_data.csv`, 404 rows;
  `integrity_report.docx`).
- **Task 3**: Neighborhood fairness comparison completed; no neighborhood found to
  be under-represented (`fairness_reflection.docx`).